<a href="https://colab.research.google.com/github/bishamkumar12/disease-classification-model/blob/main/B_Disease_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -U --quiet transformers==4.45.2 datasets evaluate seqeval accelerate
import transformers
print("Transformers version:", transformers.__version__)


In [ ]:
# Clone the repo (your link)
!git clone https://github.com/spyysalo/ncbi-disease.git
!ls -la ncbi-disease/conll


In [ ]:
from pathlib import Path

def read_conll(path):
    sents = []
    tags = []
    tokens = []
    labels = []
    with open(path, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if not line:
                if tokens:
                    sents.append(tokens)
                    tags.append(labels)
                    tokens = []
                    labels = []
                continue
            parts = line.split()
            # token is first column, tag likely last
            token = parts[0]
            tag = parts[-1]
            tokens.append(token)
            labels.append(tag)
        if tokens:
            sents.append(tokens)
            tags.append(labels)
    return sents, tags

conll_dir = Path("ncbi-disease/conll")
files = sorted([p for p in conll_dir.iterdir() if p.is_file()])
print("Files found:", [p.name for p in files])

# heuristics to find train/dev/test
train_file = dev_file = test_file = None
for p in files:
    name = p.name.lower()
    if "train" in name and train_file is None: train_file = p
    if ("dev" in name or "valid" in name) and dev_file is None: dev_file = p
    if "test" in name and test_file is None: test_file = p

# fallback: first 3 files
if not (train_file and dev_file and test_file):
    if len(files) >= 3:
        train_file, dev_file, test_file = files[0], files[1], files[2]
    else:
        raise SystemExit("Couldn't auto-detect train/dev/test — check ncbi-disease/conll")

print("Using:", train_file.name, dev_file.name, test_file.name)

train_sents, train_tags = read_conll(train_file)
dev_sents, dev_tags = read_conll(dev_file)
test_sents, test_tags = read_conll(test_file)
print("Samples:", len(train_sents), len(dev_sents), len(test_sents))


In [ ]:
from collections import OrderedDict
from datasets import Dataset, DatasetDict

# collect unique labels (keep the order)
unique_labels = list(OrderedDict.fromkeys([tag for seq in (train_tags+dev_tags+test_tags) for tag in seq]))
print("Unique labels (example):", unique_labels[:10])

# map labels->ids
label_list = unique_labels
label_to_id = {l:i for i,l in enumerate(label_list)}
id_to_label = {i:l for l,i in label_to_id.items()}
print("Label mapping:", label_to_id)

# Prepare dicts for HF datasets
def to_dataset_dict(sents, tags):
    return {"tokens": sents, "ner_tags": [[label_to_id[t] for t in seq] for seq in tags]}

train_dict = to_dataset_dict(train_sents, train_tags)
dev_dict = to_dataset_dict(dev_sents, dev_tags)
test_dict = to_dataset_dict(test_sents, test_tags)

dataset_dict = DatasetDict({
    "train": Dataset.from_dict(train_dict),
    "validation": Dataset.from_dict(dev_dict),
    "test": Dataset.from_dict(test_dict),
})
print(dataset_dict)


In [ ]:
from transformers import AutoTokenizer

# ✅ Choose a stronger pretrained model for biomedical NER
# You can switch between models below if you want to compare results:
# "microsoft/BiomedNLP-PubMedBERT-base-uncased-abstract"  ← baseline
# "dmis-lab/biobert-base-cased-v1.1"                      ← better
# "kamalkraj/BioBERT-NER"                                 ← already fine-tuned for NER (recommended)

MODEL_NAME = "alvaroalon2/biobert_diseases_ner"   # ✅ best choice for higher F1

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)

# Adjust max_length based on GPU memory
max_length = 512  # if you have enough VRAM, you can set 512

def tokenize_and_align_labels(examples):
    tokenized_inputs = tokenizer(
        examples["tokens"],
        is_split_into_words=True,
        truncation=True,
        padding="max_length",   # ✅ ensure same length for batching
        max_length=max_length,
        return_tensors=None
    )

    all_labels = examples["ner_tags"]
    new_labels = []
    for i, labels in enumerate(all_labels):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        previous_word_idx = None
        label_ids = []
        for word_idx in word_ids:
            if word_idx is None:
                label_ids.append(-100)
            elif word_idx != previous_word_idx:
                label_ids.append(labels[word_idx])
            else:
                label_ids.append(-100)
            previous_word_idx = word_idx
        new_labels.append(label_ids)

    tokenized_inputs["labels"] = new_labels
    return tokenized_inputs

# Map the tokenizer across the dataset
tokenized_datasets = dataset_dict.map(
    tokenize_and_align_labels,
    batched=True,
    remove_columns=["tokens", "ner_tags"]
)

tokenized_datasets


In [ ]:
from transformers import AutoModelForTokenClassification

num_labels = len(label_list)

model = AutoModelForTokenClassification.from_pretrained(
    "alvaroalon2/biobert_diseases_ner",
    num_labels=num_labels,
    ignore_mismatched_sizes=True  # ✅ this fixes the classifier shape mismatch
)


In [ ]:
import numpy as np
import evaluate

metric = evaluate.load("seqeval")

def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)
    true_predictions = []
    true_labels = []
    for pred_row, label_row in zip(predictions, labels):
        true_pred = []
        true_lab = []
        for p_i, l_i in zip(pred_row, label_row):
            if l_i == -100:
                continue
            true_pred.append(id_to_label[int(p_i)])
            true_lab.append(id_to_label[int(l_i)])
        true_predictions.append(true_pred)
        true_labels.append(true_lab)
    results = metric.compute(predictions=true_predictions, references=true_labels)
    # seqeval returns nested dict with overall metrics
    overall = {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"],
        "accuracy": results.get("overall_accuracy", 0.0)
    }
    return overall


In [ ]:
from transformers import TrainingArguments, Trainer, DataCollatorForTokenClassification

data_collator = DataCollatorForTokenClassification(tokenizer)

output_dir = "biobert-ner-ncbi"

training_args = TrainingArguments(
    output_dir=output_dir,
    evaluation_strategy="epoch",     # ✅ fixed typo
    save_strategy="epoch",
    learning_rate=3e-5,              # ✅ slightly higher
    warmup_ratio=0.1,
    weight_decay=0.01,
    per_device_train_batch_size=16,  # ✅ if GPU supports
    per_device_eval_batch_size=16,
    num_train_epochs=8,
    logging_dir=f"{output_dir}/logs",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    fp16=True,
    push_to_hub=False,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

trainer.train()

metrics = trainer.evaluate(tokenized_datasets["test"])
print(metrics)


In [ ]:
def predict_and_extract(sentence_tokens):
    inputs = tokenizer(sentence_tokens, is_split_into_words=True, return_tensors="pt", truncation=True)
    inputs = {k: v.to(trainer.model.device) for k, v in inputs.items()}
    outputs = trainer.model(**inputs)
    preds = outputs.logits.argmax(-1).squeeze().cpu().numpy()
    word_id_list = tokenizer(sentence_tokens, is_split_into_words=True).word_ids()
    word_preds = []
    last_word = None
    for idx, wid in enumerate(word_id_list):
        if wid is None:
            continue
        if wid != last_word:
            word_preds.append(id_to_label[int(preds[idx])])
            last_word = wid
    return list(zip(sentence_tokens, word_preds))

# Example: check a random sentence
example_sent = train_sents[10]
print("Input:", " ".join(example_sent))
print("Preds:", predict_and_extract(example_sent))


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

save_path = "/content/drive/MyDrive/pubmedbert_ncbi_disease"
trainer.save_model(save_path)
tokenizer.save_pretrained(save_path)
print("✅ Model saved to:", save_path)


In [ ]:
from transformers import AutoTokenizer, AutoModelForTokenClassification

model_path = "/content/drive/MyDrive/pubmedbert_ncbi_disease"
tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForTokenClassification.from_pretrained(model_path)

print("✅ Model loaded and ready for inference!")


In [ ]:
import numpy as np
from sklearn.metrics import classification_report

# Get predictions from the trainer
predictions, labels, _ = trainer.predict(tokenized_datasets["test"])
preds = np.argmax(predictions, axis=-1)

# Convert label IDs to label names
true_labels = [
    [id_to_label[l] for l in label if l != -100]
    for label in labels
]
pred_labels = [
    [id_to_label[p] for (p, l) in zip(pred, label) if l != -100]
    for pred, label in zip(preds, labels)
]


true_labels_flat = [item for sublist in true_labels for item in sublist]
pred_labels_flat = [item for sublist in pred_labels for item in sublist]

# Generate classification report
report = classification_report(true_labels_flat, pred_labels_flat, digits=4)
print(report)
